In [144]:
import os
import sys

PROJECT_ROOT = os.path.abspath("..")
sys.path.insert(0, PROJECT_ROOT)

In [145]:
import cv2
import mediapipe as mp
import torch
import numpy as np
from torchmetrics.text import WordErrorRate

from mediapipe.tasks import python
from mediapipe.tasks.python import vision

from srcs.datasets.transform import VideoTransform
from srcs.nets.e2e import CTCVSR, MCTCVSR, get_model
from srcs.nets.loss.ctc import ctc_decode
from srcs.nets.loss.mctc import mctc_decode
from srcs.nlp.text_transform import WordTransform, PhonemeTransform

DETECTOR_PATH = os.path.join(PROJECT_ROOT, "checkpoints", "pretrain", "blaze_face_short_range.tflite")
LANDMARKER_PATH = os.path.join(PROJECT_ROOT, "checkpoints", "pretrain", "face_landmarker.task")

CTC_CHECKPOINT = os.path.join(PROJECT_ROOT, "checkpoints", "CTCVSR", "best.pt")
MCTC_CHECKPOINT = os.path.join(PROJECT_ROOT, "checkpoints", "MCTCVSR", "best.pt")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
VIDEO_PATH = os.path.join(PROJECT_ROOT,"notebooks", "input.mp4")
LABEL = "hiện tại tôi là sinh viên"

In [146]:
word_transform = WordTransform(
    **WordTransform.vocab_paths(os.path.join(os.path.dirname(CTC_CHECKPOINT), "vocab"))
)
phoneme_transform = PhonemeTransform(
    **PhonemeTransform.vocab_paths(
        os.path.join(os.path.dirname(MCTC_CHECKPOINT), "vocab")
    )
)

word_model = (
    get_model(CTCVSR, word_transform.vocab_size, ckpt=CTC_CHECKPOINT).to(DEVICE).eval()
)
phoneme_model = (
    get_model(MCTCVSR, phoneme_transform.vocab_size, ckpt=MCTC_CHECKPOINT)
    .to(DEVICE)
    .eval()
)

wer = WordErrorRate()

In [147]:
def read_video(path):
    capture = cv2.VideoCapture(path)
    frames = []

    try:
        while True:
            success, frame = capture.read()
            if not success:
                break

            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    finally:
        capture.release()

    return frames


def crop_square(image, center, size, output_size):
    height, width = image.shape[:2]

    size = min(max(2, round(size)), height, width)

    left = int(np.clip(round(center[0] - size / 2), 0, width - size))
    top = int(np.clip(round(center[1] - size / 2), 0, height - size))

    return cv2.resize(
        image[top : top + size, left : left + size], (output_size, output_size)
    )


def crop_faces(detected):
    faces = []
    for frame, box in detected:
        center = (box.origin_x + box.width / 2, box.origin_y + box.height / 2)

        faces.append(crop_square(frame, center, max(box.width, box.height) * 1.5, 256))

    return faces

In [148]:
def detect_faces(frames):
    options = vision.FaceDetectorOptions(
        base_options=python.BaseOptions(model_asset_path=DETECTOR_PATH),
    )
    detected = []
    with vision.FaceDetector.create_from_options(options) as detector:
        for frame in frames:
            image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame)
            detections = detector.detect(image).detections

            if not detections:
                continue

            box = max(
                detections,
                key=lambda item: item.bounding_box.width * item.bounding_box.height,
            ).bounding_box

            detected.append((frame, box))

    return detected


def detect_landmarks(faces):
    options = vision.FaceLandmarkerOptions(
        base_options=python.BaseOptions(model_asset_path=LANDMARKER_PATH),
        num_faces=1,
    )

    detected = []

    with vision.FaceLandmarker.create_from_options(options) as landmarker:
        for face in faces:
            image = mp.Image(image_format=mp.ImageFormat.SRGB, data=face)
            result = landmarker.detect(image)
            if not result.face_landmarks:
                continue

            detected.append((face, result.face_landmarks[0]))

    return detected

In [149]:
def extract_lips(detected):
    indices = sorted(
        {
            index
            for connection in vision.FaceLandmarksConnections.FACE_LANDMARKS_LIPS
            for index in (connection.start, connection.end)
        }
    )

    lips = []

    for face, landmarks in detected:
        height, width = face.shape[:2]

        points = np.array(
            [[landmarks[i].x * width, landmarks[i].y * height] for i in indices]
        )

        minimum, maximum = points.min(axis=0), points.max(axis=0)
        center = (minimum + maximum) / 2
        size = float((maximum - minimum).max()) * 1.8

        lips.append(crop_square(face, center, size, 96))

    if not lips:
        raise ValueError("No usable lip frames found.")

    return np.stack(lips)

In [150]:
def run(video_path, model):
    frames = read_video(video_path)
    
    faces = detect_faces(frames)
    faces = crop_faces(faces)
    
    landmarks = detect_landmarks(faces)
    lip_frames = extract_lips(landmarks)
    
    videos = torch.from_numpy(lip_frames).permute(0, 3, 1, 2)
    videos = VideoTransform("test")(videos).unsqueeze(0).to(DEVICE)
    
    lengths = torch.tensor([videos.size(1)], dtype=torch.long, device=DEVICE)
    
    
    
    with torch.inference_mode():
        outputs = model(videos, lengths)
    
    return outputs, lengths

In [151]:
word_outputs, video_lengths = run(VIDEO_PATH,word_model)
word_ids = ctc_decode(word_outputs["logits"], video_lengths)[0]

word_transcript = word_transform.decode_for_metrics(word_ids)["wer"]
print("preds:", word_transcript)
print("wer:", float(wer(word_transcript,LABEL)))

preds: hiện tại tôi là sinh
wer: 0.1666666716337204


In [152]:
phoneme_outputs, video_lengths = run(VIDEO_PATH,phoneme_model)
phoneme_ids = mctc_decode(phoneme_outputs["logits"], video_lengths)[0]

phoneme_transcript = phoneme_transform.decode_for_metrics(phoneme_ids)["wer"]
print("preds:", phoneme_transcript)
print("wer:", float(wer(phoneme_transcript,LABEL)))

preds: tại
wer: 0.8333333134651184
